# 🧠 IOAI Prep — Day 1
# Vectors, Matrices, Transformations & Eigenvalues

---

**Resources:** [3Blue1Brown — Essence of Linear Algebra](https://www.youtube.com/playlist?list=PLZHQObOWTQDPD3MizzM2xVFitgF8hE_ab) · MML Ch 2.1–2.5  
**Time:** ~8 hours  
**Tools:** NumPy, Matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## SECTION 01 — What Is a Vector?

A vector is just a list of numbers — an arrow that points somewhere.  
If you write **v = [3, 2]**, that means *"go 3 steps right and 2 steps up"* from the origin.

There are **four ways** to think about vectors:

| View | Interpretation |
|------|---------------|
| 🎓 **Physics** | An arrow with direction and magnitude. Can be moved around freely. |
| 📐 **Math** | Anything that can be added and scaled following vector space rules. |
| 💻 **CS** | An ordered list of numbers. `[3, 2]` in 2D, `[1, 0, -4, 7]` in 4D. Each entry is a "feature." |
| 🤖 **ML** | A data point. Each image, sentence, or user is a vector in some high-dimensional space. |

> **For IOAI**, the CS/math hybrid view dominates.  
> A vector is a column of numbers in ℝⁿ.  
> A 28×28 grayscale image? That's a vector in **ℝ⁷⁸⁴** — just flatten all 784 pixels into one list.

### Notation

Vectors are written as **column vectors** by default:

```
v = ⎡ 3 ⎤
    ⎢ 2 ⎥
    ⎣-1 ⎦
```

Compactly: **v = [3, 2, -1]ᵀ** where ᵀ means transpose (flip row → column).  
The number of entries = **dimension**. This vector lives in **ℝ³**.

In [ ]:
# --- Creating vectors in NumPy ---

v = np.array([3, 2, -1])            # shape (3,) — 1D array (neither row nor column)
v_col = np.array([[3], [2], [-1]])   # shape (3,1) — actual column vector

print(f"v     = {v}  →  shape: {v.shape}")
print(f"v_col =\n{v_col}  →  shape: {v_col.shape}")

# Easier way: reshape
v_col = v.reshape(-1, 1)   # -1 means "figure it out", 1 column
print(f"\nv reshaped to column:\n{v_col}  →  shape: {v_col.shape}")

> ⚠️ **Common beginner trap:**  
> `np.array([3, 2, -1])` has shape `(3,)` — it's neither a row nor a column.  
> NumPy usually handles this fine, but in IOAI problems **shape mismatches cause silent bugs**.  
> Always check `.shape` when debugging.

---
## SECTION 02 — Vector Operations

### Addition — tip-to-tail

Place the second vector's tail at the first one's tip. Numerically: **add entry by entry**.

```
[3, 2] + [1, -1] = [3+1, 2+(-1)] = [4, 1]
```

### Scalar multiplication — stretching

Multiply every entry by the same number (the "scalar"). Stretches or flips the vector.

```
 2 · [3, 2] = [6, 4]      ← twice as long, same direction
-1 · [3, 2] = [-3, -2]    ← same length, opposite direction
```

In [ ]:
# ✏️ Exercise: Given a = [2, 5] and b = [-1, 3]

a = np.array([2, 5])
b = np.array([-1, 3])

print(f"a + b   = {a + b}")        # [1, 8]
print(f"3a      = {3 * a}")        # [6, 15]
print(f"2a - b  = {2 * a - b}")    # [5, 7]

# Magnitude (length / norm): ||a|| = sqrt(a₁² + a₂² + ... + aₙ²)
print(f"||a||   = {np.linalg.norm(a):.4f}")   # sqrt(4 + 25) ≈ 5.3852

### Dot product — how similar are two vectors?

```
a · b = a₁b₁ + a₂b₂ + ... + aₙbₙ

[2, 5] · [-1, 3] = (2)(−1) + (5)(3) = −2 + 15 = 13
```

**Geometric meaning:** `a · b = ‖a‖ · ‖b‖ · cos(θ)`

| Dot product | Meaning |
|-------------|------------------------------------------|
| **Positive** | Vectors point in roughly the same direction (θ < 90°) |
| **Zero** | Vectors are **perpendicular** (orthogonal) — huge in ML |
| **Negative** | Vectors point in roughly opposite directions (θ > 90°) |

> **Why IOAI cares:** The dot product is the foundation of ALL of ML.  
> Neural networks = dot products of inputs and weights.  
> Cosine similarity = normalized dot product.  
> Attention in transformers = dot products of query and key vectors.

In [ ]:
# --- Three ways to compute a dot product ---
print(f"np.dot(a, b) = {np.dot(a, b)}")     # 13
print(f"a @ b        = {a @ b}")             # 13  (preferred)
print(f"sum(a * b)   = {np.sum(a * b)}")     # 13

# --- Checking orthogonality ---
u = np.array([1, 0])
v = np.array([0, 1])
print(f"\nu · v = {u @ v}")   # 0 → perpendicular!

# --- Cosine similarity ---
def cosine_sim(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"cosine_sim(a, b) = {cosine_sim(a, b):.4f}")   # 0.7634

---
## SECTION 03 — Linear Combinations, Span & Independence

### Linear combinations

Scale some vectors and add them: `c₁v₁ + c₂v₂ + ... + cₖvₖ`

In [ ]:
# Standard basis vectors in ℝ²
v1 = np.array([1, 0])   # î
v2 = np.array([0, 1])   # ĵ

# 3v₁ + 2v₂ = 3[1,0] + 2[0,1] = [3, 2]
result = 3 * v1 + 2 * v2
print(f"3v₁ + 2v₂ = {result}")

# ANY vector [a, b] = a·[1,0] + b·[0,1]
# → every 2D vector is a linear combination of these two basis vectors!

### Span — what can you reach?

The **span** of a set of vectors = the set of ALL possible linear combinations.

| Setup | Span |
|-------|------|
| 1 vector | A **line** through the origin |
| 2 independent vectors | A **plane** |
| 2 parallel vectors | Still just a **line** — the second is redundant |
| 3 vectors in ℝ³ | All of 3D (if independent), a plane, or a line |

### Linear independence — no redundancy

Vectors are **linearly independent** if none of them can be written as a combination of the others.  
Equivalently: `c₁v₁ + c₂v₂ + ... + cₖvₖ = 0  ⟹  c₁ = c₂ = ... = cₖ = 0`

### Basis — the minimal spanning set

A **basis** = a set of linearly independent vectors that spans the entire space.  
In ℝⁿ you need exactly **n** independent vectors.

> **IOAI connection — PCA:** Finds a **new basis** where the first vector captures the most variance,  
> the second the next most, etc. Keep only the first k → dimensionality reduction.

---
## SECTION 04 — What Is a Matrix?

A matrix is a rectangular grid of numbers. An **m×n** matrix has m rows and n columns:

```
A = ⎡ 1   2   3 ⎤
    ⎣ 4   5   6 ⎦     ← 2×3 matrix
```

> **THE BIG IDEA** (3B1B Video 3):  
> A matrix **IS** a transformation. Its columns = where the basis vectors land.  
> Matrix × vector = applying that transformation.

### Matrix-vector multiplication

When you compute Av, you write v as a linear combination of A's columns:

```
⎡ 2   1 ⎤ ⎡ 3 ⎤ = 3·⎡ 2 ⎤ + 1·⎡ 1 ⎤ = ⎡ 7 ⎤
⎣ 0   3 ⎦ ⎣ 1 ⎦     ⎣ 0 ⎦     ⎣ 3 ⎦   ⎣ 3 ⎦
```

### Matrix-matrix multiplication

**AB means "first apply B, then apply A."** It's function composition.  
**CRITICAL:** AB ≠ BA in general! Matrix multiplication is **NOT commutative**.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
v = np.array([3, 1])

# Matrix-vector product
print(f"A @ v = {A @ v}")        # [5, 13]

# Matrix-matrix product
print(f"A @ B =\n{A @ B}")      # [[19, 22], [43, 50]]
print(f"B @ A =\n{B @ A}")      # [[23, 34], [31, 46]]  ← DIFFERENT!

# ⚠️ A * B = element-wise (Hadamard), NOT matrix multiply!
print(f"\nA * B (WRONG for matrix multiply) =\n{A * B}")

> ⚠️ **Deadly NumPy trap:** `A * B` does **element-wise** multiplication, NOT matrix multiplication.  
> Always use `A @ B` or `np.dot(A, B)` for true matrix products.

---
## SECTION 05 — Matrices as Transformations (The Geometric View)

The **single most important idea** in 3Blue1Brown's series.  
A 2×2 matrix takes every point in 2D and moves it. The columns tell you where **[1,0]** and **[0,1]** end up.

| Name | Matrix | What it does |
|------|--------|--------------|
| Identity | `[[1,0],[0,1]]` | Nothing. Every point stays put. |
| Scaling | `[[2,0],[0,3]]` | Stretch x by 2, y by 3. |
| Rotation 90° | `[[0,-1],[1,0]]` | Rotate counter-clockwise. |
| Shear | `[[1,1],[0,1]]` | Tilt space sideways. |

In [ ]:
def plot_transformation(A, title="Transformation"):
    """Visualize how matrix A transforms the unit square."""
    # Unit square corners: (0,0) → (1,0) → (1,1) → (0,1) → (0,0)
    square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]).T  # shape (2, 5)
    transformed = A @ square

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

    # Before
    ax1.fill(square[0], square[1], alpha=0.3, color='blue')
    ax1.quiver(0, 0, 1, 0, angles='xy', scale=1, scale_units='xy', color='red',   label='î')
    ax1.quiver(0, 0, 0, 1, angles='xy', scale=1, scale_units='xy', color='green', label='ĵ')
    ax1.set_title("Before"); ax1.set_xlim(-2, 5); ax1.set_ylim(-2, 5)
    ax1.set_aspect('equal'); ax1.grid(True); ax1.legend()

    # After
    ax2.fill(transformed[0], transformed[1], alpha=0.3, color='orange')
    ax2.quiver(0, 0, A[0,0], A[1,0], angles='xy', scale=1, scale_units='xy', color='red',   label='Aî')
    ax2.quiver(0, 0, A[0,1], A[1,1], angles='xy', scale=1, scale_units='xy', color='green', label='Aĵ')
    ax2.set_title(title); ax2.set_xlim(-2, 5); ax2.set_ylim(-2, 5)
    ax2.set_aspect('equal'); ax2.grid(True); ax2.legend()

    plt.tight_layout()
    plt.show()

# Try it!
A = np.array([[3, 1], [0, 2]])
plot_transformation(A, "A = [[3,1],[0,2]]")

---
## SECTION 06 — Determinants — The Area Scaling Factor

The **determinant** tells you how much a transformation scales area (2D) or volume (3D).

### 2×2 determinant

```
det ⎡ a   b ⎤ = ad − bc
    ⎣ c   d ⎦
```

| det value | Meaning |
|-----------|-------------------------------------------|
| det = 6 | Areas get multiplied by 6. |
| det = 1 | Areas preserved (e.g. rotation). |
| det = -2 | Areas scale by 2, but orientation **flips**. |
| det = 0 | Space squished to lower dimension. **NOT invertible.** |

### 3×3 determinant (cofactor expansion)

```
det ⎡ a   b   c ⎤
    ⎢ d   e   f ⎥ = a(ei−fh) − b(di−fg) + c(dh−eg)
    ⎣ g   h   i ⎦
```

In [ ]:
# --- Determinants in NumPy ---

# 2×2
A = np.array([[3, 1], [0, 2]])
print(f"det(A) = {np.linalg.det(A):.1f}")   # 6.0

# 3×3: M = [[2,1,0],[0,3,1],[1,0,2]]
# = 2(3·2 − 1·0) − 1(0·2 − 1·1) + 0 = 12 + 1 = 13
M = np.array([[2, 1, 0], [0, 3, 1], [1, 0, 2]])
print(f"det(M) = {np.linalg.det(M):.1f}")   # 13.0

# Singular matrix (det = 0) — row 2 = 3 × row 1
S = np.array([[1, 2], [3, 6]])
print(f"det(S) = {np.linalg.det(S):.1f}")   # 0.0

---
## SECTION 07 — Matrix Inverse — Undoing a Transformation

The **inverse** A⁻¹ "undoes" A: &nbsp; `A · A⁻¹ = A⁻¹ · A = I`

**A matrix is invertible if and only if det(A) ≠ 0.**  
If det = 0, the transformation squishes space — you can't unsquish.

### 2×2 inverse formula (memorize this!)

```
⎡ a   b ⎤⁻¹       1     ⎡  d  −b ⎤
⎣ c   d ⎦    =  ------  ⎣ −c   a ⎦
                 ad−bc
```

Recipe: swap a↔d, negate b and c, divide everything by det.

In [ ]:
A = np.array([[3, 1], [0, 2]])

# NumPy inverse
A_inv = np.linalg.inv(A)
print(f"A⁻¹ =\n{A_inv}")

# Verify: A @ A⁻¹ should be identity
print(f"\nA @ A⁻¹ =\n{A @ A_inv}")
print(f"Is identity? {np.allclose(A @ A_inv, np.eye(2))}")

In [ ]:
# Implement 2×2 inverse from scratch
def my_inverse_2x2(A):
    a, b = A[0, 0], A[0, 1]
    c, d = A[1, 0], A[1, 1]
    det = a * d - b * c
    if abs(det) < 1e-10:
        raise ValueError("Matrix is singular (det=0), cannot invert")
    return (1 / det) * np.array([[d, -b], [-c, a]])

# Test
print(f"my_inverse_2x2(A) =\n{my_inverse_2x2(A)}")
print(f"Matches np? {np.allclose(my_inverse_2x2(A), np.linalg.inv(A))}")

---
## SECTION 08 — Column Space & Null Space

### Column space (image): where can A send things?

The **column space** of A = the set of all possible outputs Av. It's the **span of A's columns**.

**Example:** A = [[1, 2], [2, 4]]  
Column 2 = 2 × Column 1 → column space = span{[1, 2]} = a **line**.

### Null space (kernel): what gets sent to zero?

The **null space** = all vectors v where Av = 0.  
These get "crushed to nothing" by the transformation.

**Example:** For A = [[1, 2], [2, 4]], solve Av = 0:  
x + 2y = 0 → x = −2y → null space = span{[-2, 1]} = a **line**.

> **Rank-Nullity Theorem:** `dim(column space) + dim(null space) = # columns`  
> For our example: rank 1 + nullity 1 = 2 columns. ✓  
> This is **guaranteed** to appear on IOAI theory exams.

---
## SECTION 09 — Rank

**Rank** = dimension of column space = number of linearly independent columns.  
It tells you the "true dimensionality" of the output.

| Matrix | Rank | Meaning |
|--------|------|---------|
| 2×2, full rank | 2 | Maps 2D → 2D. det ≠ 0. Invertible. |
| 2×2, rank 1 | 1 | Maps 2D → a line. det = 0. |
| 3×3, rank 2 | 2 | Maps 3D → a plane. |

In [ ]:
# Rank in NumPy

# Row 2 = 2 × Row 1, so rank < 3
M = np.array([[1, 2, 3], [2, 4, 6], [1, 1, 1]])
print(f"rank(M) = {np.linalg.matrix_rank(M)}")   # 2

# Full rank (identity)
F = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
print(f"rank(I) = {np.linalg.matrix_rank(F)}")   # 3

---
## SECTION 10 — Systems of Linear Equations & Gaussian Elimination

A system of linear equations is a matrix equation **Ax = b**.  
Question: given transformation A and desired output b, what input x gives us that output?

```
x + 2y = 5       ⎡ 1   2 ⎤ ⎡ x ⎤   ⎡ 5  ⎤
3x +  y = 10  →  ⎣ 3   1 ⎦ ⎣ y ⎦ = ⎣ 10 ⎦
```

### Gaussian elimination

Use row operations to turn the matrix into upper triangular form, then back-substitute.  
Allowed ops: (1) swap rows, (2) scale a row, (3) add a multiple of one row to another.

In [ ]:
# --- Solving systems in NumPy ---

A = np.array([[1, 2], [3, 1]])
b = np.array([5, 10])
x = np.linalg.solve(A, b)
print(f"Solution: x = {x}")                       # [3. 1.]
print(f"Verify:   A @ x = b? {np.allclose(A @ x, b)}")  # True

In [ ]:
# --- Gaussian elimination from scratch ---

def gauss_solve(A, b):
    """Solve Ax = b using Gaussian elimination with partial pivoting."""
    n = len(b)
    # Build augmented matrix [A | b]
    aug = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])

    # Forward elimination
    for col in range(n):
        # Partial pivoting: swap with row that has largest |value| in this column
        max_row = col + np.argmax(np.abs(aug[col:, col]))
        aug[[col, max_row]] = aug[[max_row, col]]

        # Eliminate all rows below the pivot
        for row in range(col + 1, n):
            factor = aug[row, col] / aug[col, col]
            aug[row] -= factor * aug[col]

    # Back-substitution
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (aug[i, -1] - aug[i, :-1] @ x) / aug[i, i]
    return x

# Test: 2×2
x = gauss_solve(A, b)
print(f"gauss_solve 2x2: x = {x}")   # [3. 1.]

# Test: 3×3
A3 = np.array([[2, 1, -1], [-3, -1, 2], [-2, 1, 2]])
b3 = np.array([8, -11, -3])
x3 = gauss_solve(A3, b3)
print(f"gauss_solve 3x3: x = {x3}")  # [2. 3. -1.]
print(f"Verify: {np.allclose(A3 @ x3, b3)}")

---
## SECTION 11 — Eigenvalues & Eigenvectors — The Crown Jewel

Most vectors **change direction** when you apply a matrix. But some special vectors only get **stretched or compressed** — they stay on the same line. These are **eigenvectors**. The scaling factor is the **eigenvalue**.

```
Av = λv
```

**v** = eigenvector, **λ** (lambda) = eigenvalue.

### How to find eigenvalues: the characteristic equation

```
Av = λv  →  (A − λI)v = 0

For non-zero v to exist: det(A − λI) = 0   ← characteristic equation!
```

### 2×2 shortcut

For `A = [[a,b],[c,d]]`: &nbsp; `λ² − trace(A)·λ + det(A) = 0`  
where `trace(A) = a + d`. Then use the quadratic formula.

### Worked example: A = [[3, 1], [0, 2]]

**Step 1 — Characteristic equation:**  
`det(A − λI) = (3−λ)(2−λ) − (1)(0) = (3−λ)(2−λ) = 0`  
→ **λ₁ = 3, λ₂ = 2**

**Step 2 — Eigenvector for λ₁ = 3:**  
`(A − 3I)v = 0 → [[0,1],[0,−1]]·[x,y] = 0 → y = 0`  
→ **v₁ = [1, 0]**

**Step 3 — Eigenvector for λ₂ = 2:**  
`(A − 2I)v = 0 → [[1,1],[0,0]]·[x,y] = 0 → x = −y`  
→ **v₂ = [−1, 1]**

### What eigenvalues tell you geometrically

| λ value | Meaning |
|---------|---------|
| λ > 1 | Eigenvector gets **stretched** |
| 0 < λ < 1 | Eigenvector gets **compressed** |
| λ = 0 | Eigenvector sent to **zero** (null space). Matrix is singular. |
| λ < 0 | Eigenvector gets **flipped and scaled** |

> **Why IOAI cares:**  
> - **PCA:** Principal components = eigenvectors of the covariance matrix. Eigenvalues = variance captured.  
> - **PageRank:** Ranking = dominant eigenvector of the link matrix.  
> - **Training stability:** Eigenvalues of the Hessian too large → gradient descent diverges.  
> - **Kernel methods:** Valid kernel ⟺ all eigenvalues of Gram matrix ≥ 0 (positive semi-definite).

In [ ]:
# --- Eigenvalues & eigenvectors in NumPy ---

A = np.array([[3, 1], [0, 2]])

eigenvalues, eigenvectors = np.linalg.eig(A)

print(f"Eigenvalues:  {eigenvalues}")     # [3. 2.]
print(f"Eigenvectors (columns):\n{eigenvectors}")
# Note: NumPy normalizes eigenvectors to unit length

In [ ]:
# --- Verify: A @ v = λ * v ---

for i in range(2):
    v = eigenvectors[:, i]     # i-th eigenvector is i-th COLUMN
    lam = eigenvalues[i]
    print(f"λ{i+1} = {lam:.1f}")
    print(f"  A @ v  = {A @ v}")
    print(f"  λ · v  = {lam * v}")
    print(f"  Match? {np.allclose(A @ v, lam * v)}\n")

In [ ]:
# --- Full visualizer: transformation + eigenvectors ---

def full_visualizer(A):
    """Visualize matrix transformation + eigenvectors."""
    square = np.array([[0,1,1,0,0], [0,0,1,1,0]])
    transformed = A @ square
    vals, vecs = np.linalg.eig(A)

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    for ax in axes:
        ax.set_xlim(-4, 6); ax.set_ylim(-4, 6)
        ax.set_aspect('equal'); ax.grid(True, alpha=0.3)
        ax.axhline(y=0, color='gray', lw=0.5)
        ax.axvline(x=0, color='gray', lw=0.5)

    # Before
    axes[0].fill(square[0], square[1], alpha=0.2, color='steelblue')
    axes[0].plot(square[0], square[1], 'steelblue', lw=2)
    axes[0].set_title('Original Unit Square', fontsize=14)

    # After + eigenvectors
    axes[1].fill(transformed[0], transformed[1], alpha=0.2, color='coral')
    axes[1].plot(transformed[0], transformed[1], 'coral', lw=2)

    colors = ['#2ecc71', '#9b59b6']
    for i in range(len(vals)):
        if np.isreal(vals[i]):
            v = vecs[:, i].real
            lam = vals[i].real
            axes[1].annotate('', xy=v*lam, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=colors[i], lw=2.5))
            axes[1].annotate('', xy=v*3, xytext=v*-3,
                arrowprops=dict(arrowstyle='-', color=colors[i], lw=1, ls='--'))
            axes[1].text(v[0]*lam+0.2, v[1]*lam+0.2,
                f'λ={lam:.1f}', color=colors[i], fontsize=12, fontweight='bold')

    axes[1].set_title(f'After A · det={np.linalg.det(A):.1f}', fontsize=14)
    plt.tight_layout()
    plt.show()

# Run it!
full_visualizer(np.array([[3, 1], [0, 2]]))

In [ ]:
# Try more matrices to build intuition:
full_visualizer(np.array([[2, 0], [0, 0.5]]))   # Scaling
full_visualizer(np.array([[0, -1], [1, 0]]))     # Rotation (complex eigenvalues!)
full_visualizer(np.array([[1, 1], [0, 1]]))      # Shear

---
---
## 🔧 My Own Code Below

Everything from here I coded myself — exploring, experimenting, and debugging.

In [ ]:
import numpy as np

# Solving systems in NumPy
A = np.array([[1, 2], [3, 1]])
b = np.array([5, 10])
x = np.linalg.solve(A, b)
print(x)                      # [3. 1.]
print(np.allclose(A @ x, b))  # True

# YOUR TASK: Implement Gaussian elimination from scratch
def gauss_solve(A, b):
    """Solve Ax = b using Gaussian elimination with back-substitution."""
    n = len(b)
    # Build augmented matrix [A | b]
    aug = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])
    
    # Forward elimination
    for col in range(n):
        # Find pivot (row with largest absolute value in this column)
        max_row = col + np.argmax(np.abs(aug[col:, col]))
        aug[[col, max_row]] = aug[[max_row, col]]  # swap rows
        
        # Eliminate below
        for row in range(col + 1, n):
            factor = aug[row, col] / aug[col, col]
            aug[row] -= factor * aug[col]
    
    # Back-substitution
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (aug[i, -1] - aug[i, :-1] @ x) / aug[i, i]
    return x

# Test
x = gauss_solve(A, b)
print(x)  # [3. 1.]

In [ ]:
print(A.shape)
print(A)
print(b.reshape(-1,1))

aug = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])
print(aug)

In [ ]:
import numpy as np
A = np.array([[2, 1, -1],
              [-3, -1, 2],
              [-2, 1, 2]])
b = np.array([8, -11, -3])

x = np.linalg.solve(A, b)

print(np.allclose(A @ x, b))

In [ ]:
def gauss_sove(A, b):
    n = len(b)

    aug = np.hstack([A.astype(float), b.reshape(-1,1).astype(float)])
    

In [ ]:
A = np.array([[1, 1, 1],
                [2, 3, 1],
                [1, 1, 2]])
b = np.array([6, 14, 9])

aug_hand = np.array([[1, 1, 1, 6],
                [2, 3, 1, 14],
                [1, 1, 2, 9]])

aug = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])
aug

In [ ]:
n = len(b)

for i in range(1, 2):
    print(i)

In [ ]:

def gauss_solve(A,b):
    n = len(b)

    aug = np.hstack([A.astype(float), b.reshape(-1,1).astype(float)])
    
    for col in range(n):
        max_row = col + np.argmax(np.abs(aug[col:, col]))
        aug[[col, max_row]] = aug[[max_row, col]]

        for row in range(col+1, n):
            factor = aug[row, col] / aug[col, col]
            aug[row] -= factor * aug[col]

    x = np.zeros(n)
    for i in range(n-1, -1, -1):
        x[i] = (aug[i, -1] - aug[i, :-1] @ x) / aug[i, i]
    return x
    
A = np.array([[2, 1, -1],
                [-3, -1, 2],
                [-2, 1, 2]])
b = np.array([8, -11, -3])
x = gauss_solve(A, b)
print(x)

In [ ]:
A = np.array([[1, 2, 3],
                [4, 5, 6],
                [7, 8, 9]])

print(A[[0, 1]])
# A[[0, 1]] = A[[1, 0]]
# print(A[[0, 1]])
print(len(A))

In [ ]:
x = np.zeros(3)
x

In [ ]:
n=2
for i in range(n - 1, -1, -1):
    print(i)